In [ ]:
# DKT Training Pipeline
# This notebook runs the complete DKT training workflow

import torch
print('CUDA available:', torch.cuda.is_available())
print('PyTorch version:', torch.__version__)

CUDA available: True


In [ ]:
# Step 1: Preprocess the merged CSV data
# This creates train/valid/test splits and saves them as pickles

!python dkt_preprocess_merged.py \
  --input ../merge_script/merged_student_question_history.csv \
  --output_dir dkt_processed \
  --min_seq_len 3 \
  --train_ratio 0.8 \
  --valid_ratio 0.1 \
  --test_ratio 0.1 \
  --seed 42

Reading input CSV from merged_student_question_history.csv ...
Detecting question indices from qN_id columns ...
Found 305 question slots: from q1 to q305
Extracting sequences per user (this may take a moment) ...
  processed 500 rows ...
  processed 1000 rows ...
  processed 1500 rows ...
Kept 1590 users with seq_len >= 3.
Building question id to index mapping ...
Number of unique questions: 95
Saving train split to dkt_processed/dkt_train.pkl ...
Saving valid split to dkt_processed/dkt_valid.pkl ...
Saving test split to dkt_processed/dkt_test.pkl ...
Saving metadata to dkt_processed/dkt_metadata.json ...
Saving question id mapping to dkt_processed/dkt_question_mapping.json ...
Done.


In [ ]:
# Step 2: Train DKT model
# This trains the model and evaluates on test set
# Automatically includes:
#   - Standard evaluation (all timesteps)
#   - Threshold optimization on validation set (always on)
#   - Test evaluation with optimized threshold
#   - Random stratified sampling evaluation

!python train_dkt_merged.py \
  --data_dir dkt_processed \
  --max_len 310 \
  --hidden 64 \
  --dropout 0.2 \
  --batch_size 64 \
  --epochs 50 \
  --lr 1e-3 \
  --device cuda \
  --save_best dkt_best.pt

Using device: cuda
Metadata: num_questions=95, total_q_slots=305.0
Epoch 1/50
  [Train] loss=0.6051, AUC=0.6635, ACC=0.7119, F1=0.8070
  Fairness per completion-rate bin:
    Bin 1 (10- 20%): students=70, predictions=2350, TPR=0.944, FPR=0.707, ACC=0.700
    Bin 2 (20- 30%): students=1, predictions=68, TPR=0.942, FPR=0.500, ACC=0.838
    Bin 3 (30- 40%): students=4, predictions=408, TPR=0.945, FPR=0.638, ACC=0.779
    Bin 4 (40- 50%): students=53, predictions=6975, TPR=0.924, FPR=0.614, ACC=0.802
    Bin 5 (50- 60%): students=15, predictions=2479, TPR=0.936, FPR=0.609, ACC=0.797
    Bin 6 (60- 70%): students=10, predictions=2005, TPR=0.942, FPR=0.640, ACC=0.778
    Bin 7 (70- 80%): students=3, predictions=705, TPR=0.940, FPR=0.733, ACC=0.696
    Bin 8 (80- 90%): students=2, predictions=541, TPR=0.951, FPR=0.612, ACC=0.793
    Bin 9 (90-100%): students=1, predictions=299, TPR=0.919, FPR=0.693, ACC=0.739
  Equalized odds distance (lowest vs highest non-empty bin): 0.0283
  Accuracy varia

In [ ]:
# Optional: Load and inspect the best model

import torch
import pickle

# Load best model checkpoint
checkpoint = torch.load('dkt_best.pt')
print("Best model info:")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Valid AUC: {checkpoint['valid_auc']:.4f}")
print(f"  Hyperparameters: {checkpoint['args']}")

# Load metadata
with open('dkt_processed/metadata.json', 'r') as f:
    import json
    metadata = json.load(f)
    print(f"\nDataset info:")
    print(f"  Number of questions: {metadata['num_questions']}")
    print(f"  Train students: {metadata['num_train']}")
    print(f"  Valid students: {metadata['num_valid']}")
    print(f"  Test students: {metadata['num_test']}")

## Alternative Configurations

### Quick Test Run (faster, for debugging)
```bash
python train_dkt_merged.py \
  --data_dir dkt_processed \
  --max_len 310 \
  --hidden 64 \
  --epochs 5 \
  --batch_size 32 \
  --device cuda
```

### High-Capacity Model (like SAKT/AKT/UKT)
```bash
python train_dkt_merged.py \
  --data_dir dkt_processed \
  --max_len 300 \
  --hidden 128 \
  --dropout 0.1 \
  --batch_size 64 \
  --epochs 50 \
  --lr 1e-3 \
  --device cuda
```

### CPU Training (no GPU)
```bash
python train_dkt_merged.py \
  --data_dir dkt_processed \
  --max_len 310 \
  --hidden 64 \
  --batch_size 32 \
  --epochs 50 \
  --device cpu
```